# Part 3: Ranking & Filtering

#### Imports

In [1]:
import nltk
import json
from collections import defaultdict
from array import array
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
import math
import numpy as np
import collections
from numpy import linalg as la
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# from wordcloud import WordCloud
from collections import Counter

#### Useful code from part 1 & part 2

In [2]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
def remove_punctuation(text):
    cleaned = ""
    for char in text:
        if char.isalnum() or char.isspace() or char == "-":
            cleaned += char
        else:
            cleaned += " "
    return cleaned


In [4]:
products_path = '../../data/fashion_products_dataset.json'
with open(products_path, "r", encoding="utf-8") as f:
    products = pd.read_json(products_path)

def build_terms(line):
    """
    Preprocess a line:
    ●  Removing stop words 
    ●  Tokenization 
    ●  Removing punctuation marks 
    ●  Stemming 
    ●  Transforming to lowercase

    Argument:
    line -- string (text) to be preprocessed

    Returns:
    line - a list of tokens corresponding to the input text after the preprocessing
    """

    stemmer = PorterStemmer()
    stop_words = set(stopwords.words("english"))
    line = line.lower()
    line = remove_punctuation(line)
    line = line.split()
    line = [x for x in line if x not in stop_words]
    line = [stemmer.stem(word) for word in line]
    return line

def get_products_information(products_df):
    elements = ["pid", "title", "description", "brand", "category", "sub_category", 
                "product_details", "seller", "out_of_stock", "selling_price", 
                "discount", "actual_price", "average_rating", "url"]
    
    products_df = products_df[elements]
    
    return products_df

products = get_products_information(products)
products["processed_title"] = products["title"].apply(build_terms)
products["processed_description"] = products["description"].apply(build_terms)
products['cat_subcat'] = products['category'] + ": " + products['sub_category']

In [5]:
products["title_description"] = products["processed_title"] + products["processed_description"]
display(products.head(5))

,pid,title,description,brand,category,sub_category,product_details,seller,out_of_stock,selling_price,discount,actual_price,average_rating,url,processed_title,processed_description,cat_subcat,title_description
0,TKPFCZ9EA7H5FYZH,Solid Women Multicolor Track Pants,Yorker trackpants made from 100% rich combed c...,York,Clothing and Accessories,Bottomwear,"[{'Style Code': '1005COMBO2'}, {'Closure': 'El...",Shyam Enterprises,False,921,69% off,"2,999",3.9,https://www.flipkart.com/yorker-solid-men-mult...,"[solid, women, multicolor, track, pant]","[yorker, trackpant, made, 100, rich, comb, cot...",Clothing and Accessories: Bottomwear,"[solid, women, multicolor, track, pant, yorker..."
1,TKPFCZ9EJZV2UVRZ,Solid Men Blue Track Pants,Yorker trackpants made from 100% rich combed c...,York,Clothing and Accessories,Bottomwear,"[{'Style Code': '1005BLUE'}, {'Closure': 'Draw...",Shyam Enterprises,False,499,66% off,"1,499",3.9,https://www.flipkart.com/yorker-solid-men-blue...,"[solid, men, blue, track, pant]","[yorker, trackpant, made, 100, rich, comb, cot...",Clothing and Accessories: Bottomwear,"[solid, men, blue, track, pant, yorker, trackp..."
2,TKPFCZ9EHFCY5Z4Y,Solid Men Multicolor Track Pants,Yorker trackpants made from 100% rich combed c...,York,Clothing and Accessories,Bottomwear,"[{'Style Code': '1005COMBO4'}, {'Closure': 'El...",Shyam Enterprises,False,931,68% off,"2,999",3.9,https://www.flipkart.com/yorker-solid-men-mult...,"[solid, men, multicolor, track, pant]","[yorker, trackpant, made, 100, rich, comb, cot...",Clothing and Accessories: Bottomwear,"[solid, men, multicolor, track, pant, yorker, ..."
3,TKPFCZ9ESZZ7YWEF,Solid Women Multicolor Track Pants,Yorker trackpants made from 100% rich combed c...,York,Clothing and Accessories,Bottomwear,"[{'Style Code': '1005COMBO3'}, {'Closure': 'El...",Shyam Enterprises,False,911,69% off,"2,999",3.9,https://www.flipkart.com/yorker-solid-men-mult...,"[solid, women, multicolor, track, pant]","[yorker, trackpant, made, 100, rich, comb, cot...",Clothing and Accessories: Bottomwear,"[solid, women, multicolor, track, pant, yorker..."
4,TKPFCZ9EVXKBSUD7,"Solid Women Brown, Grey Track Pants",Yorker trackpants made from 100% rich combed c...,York,Clothing and Accessories,Bottomwear,"[{'Style Code': '1005COMBO1'}, {'Closure': 'Dr...",Shyam Enterprises,False,943,68% off,"2,999",3.9,https://www.flipkart.com/yorker-solid-men-brow...,"[solid, women, brown, grey, track, pant]","[yorker, trackpant, made, 100, rich, comb, cot...",Clothing and Accessories: Bottomwear,"[solid, women, brown, grey, track, pant, yorke..."


## Score

#### TF-IDF

In [ ]:
def create_index_tfidf_products(products):
    index = defaultdict(list)
    tf = defaultdict(list)
    df = defaultdict(int)
    idf = defaultdict(float)
    title_index = defaultdict(str)

    num_products = len(products)

    for i in range(num_products):
        pid = products.iloc[i]["pid"]
        words = products.iloc[i]["title_description"]
        title_index[pid] = products.iloc[i].get("title", "")

        current_product_index = {}

        for position, term in enumerate(words):
            try:
                current_product_index[term][1].append(position)
            except:
                current_product_index[term] = [pid, array('I', [position])]

        norm = math.sqrt(sum(len(posting[1]) ** 2 for posting in current_product_index.values()))

        for term, posting in current_product_index.items():
            tf[term].append(np.round(len(posting[1]) / norm, 4))
            df[term] += 1

        for term, posting in current_product_index.items():
            index[term].append(posting)

    for term in df:
        idf[term] = np.round(np.log(float(num_products / df[term])), 4)

    return index, tf, df, idf, title_index


In [35]:
def rank_products(terms, docs, index, idf, tf, title_index):
    
    # I'm interested only on the element of the docVector corresponding to the query terms
    # The remaining elements would became 0 when multiplied to the query_vector
    doc_vectors = defaultdict(lambda: [0] * len(terms)) # I call doc_vectors[k] for a nonexistent key k, the key-value pair (k,[0]*len(terms)) will be automatically added to the dictionary
    query_vector = [0] * len(terms)

    # compute the norm for the query tf
    query_terms_count = collections.Counter(terms)  # get the frequency of each term in the query.
    # Example: collections.Counter(["hello","hello","world"]) --> Counter({'hello': 2, 'world': 1})
    # HINT: use when computing tf for query_vector

    query_norm = la.norm(list(query_terms_count.values()))

    for termIndex, term in enumerate(terms):  #termIndex is the index of the term in the query
        if term not in index:
            continue

        ## Compute tf*idf(normalize TF as done with documents)
        query_vector[termIndex]= query_terms_count[term]/query_norm * idf[term] #query_vector[0] corresponds to the first term in the query

        # Generate doc_vectors for matching docs
        for doc_index, (doc, postings) in enumerate(index[term]):
            # Example of [doc_index, (doc, postings)]
            # 0 (26, array('I', [1, 4, 12, 15, 22, 28, 32, 43, 51, 68, 333, 337]))
            # 1 (33, array('I', [26, 33, 57, 71, 87, 104, 109]))
            # term is in doc 26 in positions 1,4, .....
            # term is in doc 33 in positions 26,33, .....

            #tf[term][0] will contain the tf of the term "term" in the doc 26
            if doc in docs: #if the odcument is in the list of documents retrieved (matching the query)
                doc_vectors[doc][termIndex] = tf[term][doc_index] * idf[term]  # TODO: check if multiply for idf

    # Calculate the score of each doc
    # compute the cosine similarity between queyVector and each docVector:
    # HINT: you can use the dot product because in case of normalized vectors it corresponds to the cosine similarity
    # see np.dot

    doc_scores=[[np.dot(curDocVec, query_vector), doc] for doc, curDocVec in doc_vectors.items() ]
    doc_scores.sort(reverse=True)
    result_docs = [x[1] for x in doc_scores]
    #print document titles instead if document id's
    #result_docs=[ title_index[x] for x in result_docs ]
    if len(result_docs) == 0:
        print("No results found, try again")
        query = input()
        docs = search_tf_idf(query, index)
    #print ('\n'.join(result_docs), '\n')
    return result_docs

def search_tf_idf(query, index, idf, tf, title_index):
    """
    output is the list of documents that contain any of the query terms.
    So, we will get the list of documents for each query term, and take the union of them.
    """
    query = build_terms(query)

    docs = None
    
    for term in query:
        if term in index:
            # store in term_docs the ids of the docs that contain "term"
            term_docs= {posting[0] for posting in index[term]}

            if docs is None:
                docs = term_docs              
            else:
                docs &= term_docs             
        else:
            docs = set()
            break

    docs = list(docs)
    ranked_docs = rank_products(query, docs, index, idf, tf, title_index)
    return ranked_docs

In [37]:
index, tf, df, idf, title_index = create_index_tfidf_products(products)

print("Insert your query:\n")
query = input()
ranked_products = search_tf_idf(query, index, idf, tf, title_index)
top = 10

print("\n======================\nTop {} results out of {} for the query {}:\n".format(top, len(ranked_products), query))
for pid in ranked_products[:top]:
    print("product_id= {} - page_title: {}".format(pid, title_index[pid]))

Insert your query:


Top 10 results out of 4189 for the query Men Round Neck:

product_id= TSHFUNN3QMCGSNCD - page_title: Printed Men Round Neck Pink T-Shirt
product_id= TSHFUNN2WF5PB3NZ - page_title: Printed Men Round Neck White T-Shirt
product_id= TSHFUNN2H8DUMJYQ - page_title: Printed Men Round Neck Blue T-Shirt
product_id= TSHFME2ERCGZHP6Y - page_title: Solid Men Round Neck Red T-Shirt
product_id= TSHFVXGQ8Z5ZHCBS - page_title: Printed Men Round Neck Yellow T-Shirt
product_id= TSHFME2EGJMF23GF - page_title: Self Design Men Round Neck Blue T-Shirt
product_id= TSHFME2EG4KETWYS - page_title: Striped Men Round Neck White, Blue T-Shirt
product_id= TSHFWF57WJBQ4NUG - page_title: Printed Men Round Neck White T-Shirt
product_id= TSHFWF57UMK9QHDW - page_title: Solid Men Round Neck Green T-Shirt
product_id= TSHFVXGQG7SSCQFH - page_title: Printed Men Round Neck Black T-Shirt


#### BM25

In [38]:
def create_index_bm25_products(products, k1=1.5, b=0.75):
    index = defaultdict(list)          
    df = defaultdict(int)              
    tf = defaultdict(list)            
    title_index = defaultdict(str)    
    doc_len = {}                       
    N = len(products)                 

    for i in range(N):
        pid = products.iloc[i]["pid"]
        words = products.iloc[i]["title_description"]
        title_index[pid] = products.iloc[i].get("title", "")
        doc_len[pid] = len(words)

        term_positions = {}

        for pos, term in enumerate(words):
            if term in term_positions:
                term_positions[term].append(pos)
            else:
                term_positions[term] = [pos]

        for term, positions in term_positions.items():
            index[term].append([pid, array('I', positions)])
            df[term] += 1
            tf[term].append(len(positions))    

    idf = {}
    for term, freq in df.items():
        idf[term] = math.log(N / freq)

    avg_doc_len = sum(doc_len.values()) / N

    return index, tf, df, idf, title_index, doc_len, avg_doc_len, k1, b

In [39]:
def rank_products_bm25(terms, docs, index, idf, tf, doc_len, avg_doc_len, k1, b):
    scores = defaultdict(float)

    for term_index, term in enumerate(terms):
        if term not in index:
            continue

        postings = index[term]      
        tf_list = tf[term]
        idf_value = idf[term]

        for i, (doc_id, positions) in enumerate(postings):
            if doc_id not in docs:
                continue

            f = tf_list[i]          
            dl = doc_len[doc_id]    

            denom = f + k1 * (1 - b + b * dl / avg_doc_len)
            score = idf_value * ((f * (k1 + 1)) / denom)

            scores[doc_id] += score

    doc_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    result_docs = [doc for doc, score in doc_scores]
    if len(result_docs) == 0:
        print("No results found, try again")
        query = input()
        docs = search_bm25(query, index)

    return result_docs

def search_bm25(query, index, idf, tf, title_index, doc_len, avg_doc_len, k1, b):
    terms = build_terms(query)

    docs = None
    for term in terms:
        if term not in index:
            return []   

        term_docs = {posting[0] for posting in index[term]}

        if docs is None:
            docs = term_docs
        else:
            docs &= term_docs

        if not docs:
            return []   

    docs = list(docs)
    ranked_docs = rank_products_bm25(terms, docs, index, idf, tf, doc_len, avg_doc_len, k1, b)
    return ranked_docs


In [40]:
index, tf, df, idf, title_index, doc_len, avg_doc_len, k1, b = create_index_bm25_products(products)

print("Insert your query:\n")
query = input()
ranked_products = search_bm25(query, index, idf, tf, title_index, doc_len, avg_doc_len, k1, b)
top = 10

print("\n======================\nTop {} results out of {} for the query {}:\n".format(top, len(ranked_products), query))
for pid in ranked_products[:top]:
    print("product_id= {} - page_title: {}".format(pid, title_index[pid]))

Insert your query:


Top 10 results out of 4189 for the query Men Round Neck:

product_id= TSHFUNN2WF5PB3NZ - page_title: Printed Men Round Neck White T-Shirt
product_id= TSHFUNN2H8DUMJYQ - page_title: Printed Men Round Neck Blue T-Shirt
product_id= TSHFUNN3QMCGSNCD - page_title: Printed Men Round Neck Pink T-Shirt
product_id= TSHFVXGQG7SSCQFH - page_title: Printed Men Round Neck Black T-Shirt
product_id= TSHFVXGQ8Z5ZHCBS - page_title: Printed Men Round Neck Yellow T-Shirt
product_id= TSHFWF58ZNPZTXYG - page_title: Solid Men Round Neck White T-Shirt
product_id= TSHFWF589Z4GGVJ2 - page_title: Solid Men Round Neck Red T-Shirt
product_id= TSHFWF57CHKBUNHE - page_title: Solid Men Round Neck White T-Shirt
product_id= TSHFWF589JQGEZES - page_title: Solid Men Round Neck White T-Shirt
product_id= TSHFWF57WJBQ4NUG - page_title: Printed Men Round Neck White T-Shirt


#### Our own score system